# Apache Mahout Hackathon: Qumat and QDP in Google Colab

Welcome! This notebook installs Apache Mahout's `qumat` package and walks through the examples from the kickoff talk, so you can run everything in the browser.

## Before you run anything: set the runtime (1 minute)

Colab's default runtime is **Python 3.13**, and Mahout 0.6.0 supports **Python 3.10 to 3.12**. Pick the older runtime and a GPU first, otherwise the install fails and the QDP part cannot run.

1. Menu **Runtime > Change runtime type**.
2. **Runtime version**: choose **2026.07** (Python 3.12.13). Not "Latest".
3. **Hardware accelerator**: choose **T4 GPU**.
4. Click **Save**. If Colab asks to restart or reconnect, say yes.
5. Menu **Runtime > Run all**.

The first cell of Part 0 prints the Python version. It should say `3.12`; if it says `3.13`, step 2 did not stick, so do it again.

**How the notebook is organized**

- Parts 0 to 3: install and the basics. Today is a 3-hour hackathon: spend at most 10 minutes here, then pick a task from Part 7.
- Parts 4 and 5: optional extras (parameterized circuits, swap test).
- Part 6: QDP on the GPU. Needs the T4 and the 2026.07 runtime from above.
- Part 7: the task list.

**Links**

- Website and docs: https://mahout.apache.org
- Source and issues: https://github.com/apache/mahout
- Mailing list: dev@mahout.apache.org

## Part 0: Install

Apache Mahout ships on PyPI as `qumat`. The current release, 0.6.0, declares **Python 3.10 to 3.12**. Run the next cell first: it tells you which Python this runtime has and what to do if it is newer.

**If it reports Python 3.13 or newer** (the Colab default may have moved past 3.12): the clean fix is `Runtime > Change runtime type > Runtime version` and pick **2026.07** (Python 3.12.13), then `Runtime > Run all` again. Without that, pip silently falls back to the ancient `qumat 0.0.1` and tries to compile old `qiskit-aer` from source, which fails. Step 1 below has a workaround for Qumat only; the QDP wheel (Step 2) exists for 3.10 to 3.12 only.

In [ ]:
import platform
import sys

PY = sys.version_info[:2]
print("Python", platform.python_version(), "on", platform.system())
!nvidia-smi -L 2>/dev/null || echo "No GPU attached (that is fine for Parts 0-5)"

if PY > (3, 12):
    print(
        "\nThis Python is newer than qumat 0.6.0 declares. Recommended: Runtime > Change runtime type >"
        "\nRuntime version > 2026.07 (Python 3.12), then Run all again. Step 1 will otherwise install"
        "\nqumat with --ignore-requires-python; Step 2 (the QDP wheel) will be skipped."
    )

### Step 1: Qumat (pure Python, works everywhere)

In [ ]:
if PY > (3, 12):
    # All of qumat 0.6.0's dependencies ship wheels for 3.13; only qumat's own metadata says <3.13.
    %pip install -q --ignore-requires-python "qumat==0.6.0"
else:
    %pip install -q qumat

### Step 2: QDP, the GPU extra

`pip install "qumat[qdp]"` adds the `qumat-qdp` package: a prebuilt Linux wheel for Python 3.10 to 3.12 that bundles the Rust engine and CUDA kernels. It installs on any Linux machine, including a CPU-only Colab runtime, but it needs an NVIDIA GPU to actually encode anything (Part 6). There are no macOS, Windows or Python 3.13 wheels, so the cell skips itself in those cases.

Changing QDP's Rust or CUDA code means rebuilding it from source; the steps are in the appendix of `tasks.md`, not needed today.

In [ ]:
if platform.system() != "Linux":
    print("qumat-qdp only ships Linux wheels; skipping on", platform.system())
elif PY > (3, 12):
    print("qumat-qdp has no wheel for Python", platform.python_version(), "- pin the runtime version to 2026.07 to use Part 6")
else:
    %pip install -q "qumat[qdp]"

If Colab shows a *Restart runtime* button after the installs, click it and continue from the next cell. The packages stay installed.

In [ ]:
from importlib.metadata import PackageNotFoundError, version

from qumat import QuMat

print("qumat    ", version("qumat"))
try:
    print("qumat-qdp", version("qumat-qdp"), "(GPU extra installed; needs a GPU at run time)")
except PackageNotFoundError:
    print("qumat-qdp not installed (Parts 1-5 do not need it)")

## Part 1: Your first circuit, a Bell state

`QuMat` is the one class you need. You give it a backend config (which simulator to use and how many shots), then build the circuit gate by gate.

A Bell state is the "hello world" of quantum computing: a Hadamard puts qubit 0 into superposition, and a CNOT entangles qubit 1 with it. Measuring should give only `00` and `11`, each about half the time.

In [ ]:
qiskit_config = {
    "backend_name": "qiskit",
    "backend_options": {"simulator_type": "aer_simulator", "shots": 1024},
}

qc = QuMat(qiskit_config)
qc.create_empty_circuit(num_qubits=2)
qc.apply_hadamard_gate(0)
qc.apply_cnot_gate(0, 1)

print(qc.draw())
counts = qc.execute_circuit()
print(counts)

In [ ]:
import matplotlib.pyplot as plt

plt.bar(counts.keys(), counts.values(), color="#F28C28")
plt.title("Bell state, 1024 shots")
plt.xlabel("measured bit string")
plt.ylabel("counts")
plt.show()

**Try it:** add `qc.apply_pauli_x_gate(0)` before executing and see how the histogram changes.

## Part 2: Same circuit, three backends

The backend is just a config dictionary. The same gate calls run on Qiskit Aer, Cirq, or the Amazon Braket local simulator.

In [ ]:
def bell(backend_name, simulator_type, shots=100):
    qc = QuMat({
        "backend_name": backend_name,
        "backend_options": {"simulator_type": simulator_type, "shots": shots},
    })
    qc.create_empty_circuit(2)
    qc.apply_hadamard_gate(0)
    qc.apply_cnot_gate(0, 1)
    return qc.execute_circuit()


for name, sim in [("qiskit", "aer_simulator"), ("cirq", "default"), ("amazon_braket", "local")]:
    print(f"{name:14s} -> {bell(name, sim)}")

Notice the result formats differ slightly: Qiskit and Braket return a dict of bit strings, Cirq returns a list with integer basis states (`3` is `11`). A small helper that normalizes these would make a nice hackathon contribution.

## Part 3: Look at the state vector

Counts are samples. `get_final_state_vector()` returns the exact amplitudes, which is handy for debugging and for teaching.

For the Bell state you should see amplitude `1/sqrt(2) ~ 0.707` on `|00>` and `|11>` and zero elsewhere.

In [ ]:
import numpy as np

qc = QuMat(qiskit_config)
qc.create_empty_circuit(2)
qc.apply_hadamard_gate(0)
qc.apply_cnot_gate(0, 1)

state = np.asarray(qc.get_final_state_vector())
for index, amplitude in enumerate(state):
    print(f"|{index:02b}>  amplitude = {np.round(amplitude, 3)}   probability = {abs(amplitude) ** 2:.3f}")

## Part 4: Parameterized circuits and a tiny optimization

Rotation gates accept a **parameter name** instead of a number. You bind values when you execute, which is exactly what variational algorithms (VQE, QAOA, quantum ML) need.

First, sweep `theta` on an `Rx` gate. Theory says `P(1) = sin^2(theta / 2)`.

In [ ]:
qc = QuMat(qiskit_config)
qc.create_empty_circuit(1)
qc.apply_rx_gate(0, "theta")  # symbolic parameter

thetas = np.linspace(0, 2 * np.pi, 13)
measured = []
for theta in thetas:
    counts = qc.execute_circuit(parameter_values={"theta": float(theta)})
    measured.append(counts.get("1", 0) / 1024)

plt.plot(thetas, np.sin(thetas / 2) ** 2, label="theory: sin^2(theta/2)", color="#1FA3A3")
plt.plot(thetas, measured, "o", label="measured (1024 shots)", color="#F28C28")
plt.xlabel("theta")
plt.ylabel("P(1)")
plt.legend()
plt.show()

Now let an optimizer drive the parameter. The cost function runs the circuit and returns how far `P(1)` is from a target of 0.25. SciPy's COBYLA is gradient-free, so shot noise is fine.

In [ ]:
from scipy.optimize import minimize

TARGET = 0.25
history = []


def cost(x):
    counts = qc.execute_circuit(parameter_values={"theta": float(x[0])})
    p1 = counts.get("1", 0) / 1024
    history.append(p1)
    return (p1 - TARGET) ** 2


result = minimize(cost, x0=[0.1], method="COBYLA", options={"maxiter": 40, "rhobeg": 0.5})
print(f"best theta = {result.x[0]:.3f}  (theory: {2 * np.arcsin(np.sqrt(TARGET)):.3f})")
print(f"P(1) at best theta = {history[-1]:.3f}")

plt.plot(history, color="#F28C28")
plt.axhline(TARGET, linestyle="--", color="#1FA3A3", label="target")
plt.xlabel("evaluation")
plt.ylabel("P(1)")
plt.legend()
plt.show()

## Part 5: Swap test, how similar are two states?

Qumat has a built-in swap test. `measure_overlap(q1, q2, ancilla_qubit)` returns the fidelity `|<psi|phi>|^2` between the states on two qubits.

- `|0>` vs `|0>` should give about `1.0`
- `|0>` vs `|+>` (after a Hadamard) should give about `0.5`
- `|0>` vs `|1>` should give about `0.0`

In [ ]:
def overlap(prepare_second):
    qc = QuMat({
        "backend_name": "qiskit",
        "backend_options": {"simulator_type": "aer_simulator", "shots": 4096},
    })
    qc.create_empty_circuit(3)   # qubit 0 = ancilla, qubit 1 = |0>, qubit 2 = state under test
    prepare_second(qc)
    return qc.measure_overlap(1, 2, ancilla_qubit=0)


print("|0> vs |0> :", round(overlap(lambda qc: None), 3))
print("|0> vs |+> :", round(overlap(lambda qc: qc.apply_hadamard_gate(2)), 3))
print("|0> vs |1> :", round(overlap(lambda qc: qc.apply_pauli_x_gate(2)), 3))

## Part 6 (GPU): QDP, encoding classical data on the GPU

QDP (Quantum Data Plane) is the second half of Mahout. It is a Rust engine with CUDA kernels that turns classical data into quantum state vectors directly on the GPU and hands them to PyTorch through DLPack with no host copy.

**This part needs a GPU runtime and a Python 3.10 to 3.12 runtime** (the `qumat-qdp` wheel has no 3.13 build yet). On the free tier pick `Runtime > Change runtime type > T4 GPU` and, if Part 0 reported Python 3.13, `Runtime version > 2026.07`. The cells below skip themselves otherwise.

In [ ]:
try:
    import torch

    HAS_GPU = torch.cuda.is_available()
except ImportError:  # torch is preinstalled on Colab; this only matters elsewhere
    HAS_GPU = False

try:
    import _qdp  # the native extension inside the qumat-qdp wheel

    HAS_QDP = True
except ImportError:
    HAS_QDP = False

QDP_OK = HAS_GPU and HAS_QDP
print("CUDA available:", HAS_GPU, "| qumat-qdp installed:", HAS_QDP)
if not HAS_GPU:
    print("Switch to a GPU runtime and re-run from Part 0 to try QDP. The cells below skip themselves.")
elif not HAS_QDP:
    print(
        "GPU found, but the qumat-qdp wheel is not installed for Python", platform.python_version() + "."
        "\nWheels exist for 3.10 to 3.12 only: Runtime > Change runtime type > Runtime version > 2026.07,"
        "\nthen re-run from Part 0. The cells below skip themselves."
    )

QDP itself was installed in Part 0 (`pip install "qumat[qdp]"`). Encode four numbers into a 2-qubit state and check the amplitudes against NumPy.

In [ ]:
if QDP_OK:
    from qumat.qdp import QdpEngine

    engine = QdpEngine(device_id=0)  # CUDA backend by default

    data = [1.0, 2.0, 3.0, 4.0]
    qtensor = engine.encode(data, num_qubits=2, encoding_method="amplitude")

    # DLPack hand-off: single use, so convert once and keep the tensor.
    state = torch.from_dlpack(qtensor)
    print("shape:", tuple(state.shape), "dtype:", state.dtype, "device:", state.device)

    expected = np.array(data) / np.linalg.norm(data)
    print("GPU amplitudes :", state[0].real.cpu().numpy().round(4))
    print("numpy reference:", expected.round(4))

Batches are where QDP shines. Encode a few thousand samples at once and time it.

In [ ]:
if QDP_OK:
    import time

    NUM_QUBITS = 10            # 1024 amplitudes per sample
    BATCH = 4096
    batch = torch.rand(BATCH, 2 ** NUM_QUBITS, dtype=torch.float32, device="cuda")

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    qtensor = engine.encode(batch, num_qubits=NUM_QUBITS, encoding_method="amplitude")
    states = torch.from_dlpack(qtensor)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    print("encoded:", tuple(states.shape), states.dtype, states.device)
    print(f"{BATCH} samples in {elapsed * 1000:.1f} ms  ({BATCH / elapsed:,.0f} samples/s)")

    # Each row is a normalized quantum state.
    norms = states.abs().pow(2).sum(dim=1)
    print("row norms (should all be ~1):", norms[:5].cpu().numpy().round(4))

Other encoders use the same call: `encoding_method="angle"` (one angle per qubit), `"basis"` (an integer index), `"phase"`, `"iqp"` and `"iqp-z"`. The encoded tensor can go straight into a PyTorch model; see `examples/qdp/simple.ipynb` in the repository for a full training loop.

## Part 7: Hackathon tasks and how to contribute

Today is a **3-hour** hackathon, so every task below is sized for one to two hours, needs no GPU, and ends in a PR with a test. The full sheet with reproduction steps and ready-to-file issue text is `tasks.md` in the hackathon kit; bigger follow-ups are in its appendix.

**Track A: Qumat (Python only).** All reproduced on `main` and on `qumat` 0.6.0.

| # | Issue | Task |
|---|---|---|
| A1 | [#1506](https://github.com/apache/mahout/issues/1506) | Clear `ValueError` for an unknown backend name (today a `ModuleNotFoundError` leaks) |
| A2 | [#1507](https://github.com/apache/mahout/issues/1507) | Unsupported `simulator_type` should raise instead of `print()` and silently continuing |
| A3 | [#1508](https://github.com/apache/mahout/issues/1508) | Cirq backend ignores the initialized simulator and creates its own `cirq.Simulator()` |
| A4 | [#1509](https://github.com/apache/mahout/issues/1509) | Bug: reusing a parameter name on two gates fails on Qiskit only (`name conflict adding parameter 't'`) |
| A5 | [#1510](https://github.com/apache/mahout/issues/1510) | `create_empty_circuit()` without `num_qubits`: Qiskit raises, Cirq and Braket allocate on demand |
| A6 | [#1511](https://github.com/apache/mahout/issues/1511) | Add **one** missing gate per person: S, S-dagger, T-dagger, CZ or Phase, on all three backends with a test |
| A7 | [#1512](https://github.com/apache/mahout/issues/1512) | Python 3.13 support: lift the `<3.13` cap in `pyproject.toml` and add 3.13 to CI (all dependencies already ship 3.13 wheels; see Part 0) |

**Track B: QDP.** B1 to B3 are pure Python. B4 to B6 are Rust: take one only if `cargo` is already installed, and start the first `cargo build` right away (5 to 10 minutes). No CUDA needed, the build uses stubs like CI does.

| # | Issue | Task |
|---|---|---|
| B1 | [#1469](https://github.com/apache/mahout/issues/1469) | `QdpBenchmark` does not validate builder argument ranges |
| B2 | [#1331](https://github.com/apache/mahout/issues/1331) | QDP tests should skip, not error, when no GPU is present |
| B3 | [#1468](https://github.com/apache/mahout/issues/1468) | File loader ignores the configured batch limit (one-line wrapper fix plus a test) |
| B4 | [#1472](https://github.com/apache/mahout/issues/1472) | Read default LZ4-compressed Feather files (Rust reader) |
| B5 | [#787](https://github.com/apache/mahout/issues/787) | NumPy file size check to prevent OOM (Rust reader) |
| B6 | [#1436](https://github.com/apache/mahout/issues/1436) | `StreamingProducer`: `VecDeque` for O(1) buffer advance (Rust) |

**Setup** (in a terminal, not in Colab; about a minute):

```bash
git clone https://github.com/<you>/mahout.git && cd mahout
pip install uv
uv sync                      # Qumat and the Python QDP package
uv run pre-commit install    # ruff + type checks on every commit
uv run pytest testing/qumat  # smoke test
```

**Workflow:** comment on the issue to claim it, branch, make the change with a test, open a PR that links the issue. One committer approval is required to merge, so ping a mentor in the room. Discussion happens on dev@mahout.apache.org.

Have fun, and thanks for hacking on Apache Mahout!